In [ ]:
import os
import shutil

print("Setting up dataset...")

# Get the notebook directory (works for all team members)
notebook_dir = os.getcwd()
print(f"Notebook directory: {notebook_dir}")

# Look for dataset folder in Detection and Classification directory
dataset_folder = os.path.join(notebook_dir, "food-1")

# If food-1 exists and has the roboflow.zip but no train folder, copy folders
if os.path.exists(dataset_folder):
    print(f"✓ Found food-1 folder: {dataset_folder}")
    
    # Check if train/valid folders already exist
    train_exists = os.path.exists(os.path.join(dataset_folder, "train"))
    valid_exists = os.path.exists(os.path.join(dataset_folder, "valid"))
    
    if train_exists and valid_exists:
        print("✓ train/ and valid/ folders already exist in food-1")
    else:
        # Look for source folders to copy from
        print("\n→ Copying train/valid folders to food-1...")
        
        # Check in parent directory for IndianFoodNet
        parent_dir = os.path.dirname(notebook_dir)
        source_dataset = os.path.join(parent_dir, "IndianFoodNet.v1i.yolov8 (Unzipped Files)")
        
        if not os.path.exists(source_dataset):
            source_dataset = os.path.join(parent_dir, "IndianFoodNet.v1i.yolov8")
        
        if os.path.exists(source_dataset):
            print(f"  Found source at: {source_dataset}")
            
            # Copy train folder
            src_train = os.path.join(source_dataset, "train")
            dst_train = os.path.join(dataset_folder, "train")
            
            if os.path.exists(src_train) and not os.path.exists(dst_train):
                print(f"  → Copying train folder...")
                shutil.copytree(src_train, dst_train)
                print(f"  ✓ train/ copied to food-1")
            
            # Copy valid folder
            src_valid = os.path.join(source_dataset, "valid")
            dst_valid = os.path.join(dataset_folder, "valid")
            
            if os.path.exists(src_valid) and not os.path.exists(dst_valid):
                print(f"  → Copying valid folder...")
                shutil.copytree(src_valid, dst_valid)
                print(f"  ✓ valid/ copied to food-1")
        else:
            print(f"✗ Could not find IndianFoodNet source folder")

# Now set up dataset object
class DatasetPath:
    def __init__(self, path):
        self.location = path

if os.path.exists(dataset_folder):
    dataset = DatasetPath(dataset_folder)
    
    train_path = os.path.join(dataset.location, "train")
    valid_path = os.path.join(dataset.location, "valid")
    
    if os.path.exists(train_path):
        print(f"✓ Train folder: OK")
    else:
        print(f"✗ Train folder: NOT FOUND")
    
    if os.path.exists(valid_path):
        print(f"✓ Valid folder: OK")
    else:
        print(f"✗ Valid folder: NOT FOUND")
    
    print(f"\n✓ Dataset ready at: ./food-1/")
else:
    print(f"✗ food-1 folder not found in {notebook_dir}")
    dataset = None

In [ ]:
import os
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

# Paths
data_path = dataset.location

train_csv = os.path.join(data_path, "train", "_classes.csv")
val_csv = os.path.join(data_path, "valid", "_classes.csv")

train_df = pd.read_csv(train_csv)
val_df = pd.read_csv(val_csv)

class_names = train_df.columns[1:]
num_classes = len(class_names)

print("Classes:", num_classes)

In [ ]:
# Image loader (padding, not cropping)
IMG_SIZE = 224

def load_image(row, folder):

    filename = row[0]
    label = tf.argmax(row[1:])

    img_path = os.path.join(folder, filename)

    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)

    # pad to square
    h = tf.shape(img)[0]
    w = tf.shape(img)[1]
    max_dim = tf.maximum(h, w)

    img = tf.image.resize_with_pad(img, max_dim, max_dim)

    # resize to model size
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))

    # ResNet preprocessing
    img = tf.keras.applications.resnet.preprocess_input(img)

    return img, label

In [ ]:
# tf.data pipelines
BATCH_SIZE = 32

def make_dataset(df, folder, training=True):

    ds = tf.data.Dataset.from_tensor_slices(df.values)

    ds = ds.map(
        lambda row: load_image(row, folder),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    if training:
        ds = ds.shuffle(1000)

    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds

train_ds = make_dataset(train_df, os.path.join(data_path, "train"), True)
val_ds = make_dataset(val_df, os.path.join(data_path, "valid"), False)

In [ ]:
# Model (ResNet50 fine-tuning)
base_model = tf.keras.applications.ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = True

model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(num_classes, activation="softmax")
])

In [ ]:
# Compile
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
# Train
EPOCHS = 20

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

In [ ]:
# Plot accuracy
plt.figure(figsize=(8, 5))

plt.plot(history.history["accuracy"], label="Train")
plt.plot(history.history["val_accuracy"], label="Validation")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.title("Training vs Validation Accuracy")

plt.show()

In [ ]:
# Save the model
model_save_path = os.path.join(data_path, "resnet50_tf_trained.h5")
model.save(model_save_path)
print(f"✓ Model saved to: {model_save_path}")
print(f"✓ File size: {os.path.getsize(model_save_path) / (1024**2):.2f} MB")